## Transform data and load to silver Delta table

In this exercise, you will use PySpark and SQL.

In [5]:
from pyspark.sql.types import *

# create the schema for the table
orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())

])

# Import all files from Bronze folder to lakehouse
df = spark.read.format("csv").option("header", "false").schema(orderSchema).load("Files/bronze/*.csv")

# display the first 10 rows of the DataFrame to preview your data
display(df.limit(10))

StatementMeta(, 1a627465-7a9a-4581-ac13-bac6d131782b, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b223cdba-bef3-431b-9979-023da47b4b3b)

Now we will add columns for data validation and cleanup, using a PySpark dataframe to add columns and update the values of some of the existing columns.

In [6]:
from pyspark.sql.functions import when, lit, col, current_timestamp, input_file_name

# Add columns IsFlagged, CreatedTS and ModifiedTS
df = df.withColumn("FileName", input_file_name()) \
    .withColumn("IsFlagged", when(col("OrderDate") < "2019-08-01", True).otherwise(False)) \
    .withColumn("CreatedTS", current_timestamp()).withColumn("ModifiedTS", current_timestamp())

# update CustomerName to "Unknown" if CustomerName null or empty
df = df.withColumn("CustomerName", when((col("CustomerName").isNull() | (col("CustomerName") == "")), lit("Unknown")).otherwise(col("CustomerName")))


StatementMeta(, 1a627465-7a9a-4581-ac13-bac6d131782b, 8, Finished, Available, Finished, False)

Now, we will define the schema for the sales_silver table in the sales database using Delta Lake format. 

Create a new code block and add the following code to the cell,

In [7]:
# Define the schema for the sales_silver table

from pyspark.sql.types import *
from delta.tables import *

DeltaTable.createIfNotExists(spark) \
    .tableName("sales.sales_silver") \
    .addColumn("SalesOrderNumber", StringType()) \
    .addColumn("SalesOrderLineNumber", IntegerType()) \
    .addColumn("OrderDate", DateType()) \
    .addColumn("CustomerName", StringType()) \
    .addColumn("Email", StringType()) \
    .addColumn("Item", StringType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("UnitPrice", FloatType()) \
    .addColumn("Tax", FloatType()) \
    .addColumn("FileName", StringType()) \
    .addColumn("IsFlagged", BooleanType()) \
    .addColumn("CreatedTS", DateType()) \
    .addColumn("ModifiedTS", DateType()) \
    .execute()

StatementMeta(, 1a627465-7a9a-4581-ac13-bac6d131782b, 9, Finished, Available, Finished, False)

#### Now we are going to perform an **"upsert operation"** on a Delta table, updating existing records based on specific conditions and inserting new records when no match is found. Add a new code block and paste the follwoing code,

In [8]:
# update existing records and insert new ones based on a conditin defined by the columns SalesOrderNumber, OrderDate, CustomerName, and Item.

from delta.tables import *
# deltaTable = DeltaTable.forPath(spark, "Tables/sales_sliver")
deltaTable = DeltaTable.forName(spark, "sales.sales_silver")

dfUpdates = df
    
deltaTable.alias('silver').merge(dfUpdates.alias('updates'),
    'silver.SalesOrderNumber = updates.SalesOrderNumber and silver.OrderDate = updates.OrderDate and silver.CustomerName = updates.CustomerName and silver.Item = updates.Item'
  ) \
   .whenMatchedUpdate(set =
    {
          
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "SalesOrderNumber": "updates.SalesOrderNumber",
      "SalesOrderLineNumber": "updates.SalesOrderLineNumber",
      "OrderDate": "updates.OrderDate",
      "CustomerName": "updates.CustomerName",
      "Email": "updates.Email",
      "Item": "updates.Item",
      "Quantity": "updates.Quantity",
      "UnitPrice": "updates.UnitPrice",
      "Tax": "updates.Tax",
      "FileName": "updates.FileName",
      "IsFlagged": "updates.IsFlagged",
      "CreatedTS": "updates.CreatedTS",
      "ModifiedTS": "updates.ModifiedTS"
    }
  ) \
  .execute()


StatementMeta(, 1a627465-7a9a-4581-ac13-bac6d131782b, 10, Finished, Available, Finished, False)

Above operation is important because it enables you to "update existing records" in the table based on the values of specific columns, and "insert new records" when no match is found. 

This is a common requirement when you’re loading data from a source system that may contain updates to existing and new records.


You now have data in your "silver delta table" that is ready for further transformation and modeling.